# CPSA 04 — variant study (activation × training/deployment size)

Trains the variant matrix of YOLOv8n-pose on the mixed dataset: the
half-bust crops from notebook 01 plus the full-frame images from notebook 02
(`FULL_DATA`). Every row is
`(activation, train_imgsz, deploy HxW)`; the default list is a **proposal
pending approval**:

| variant | activation | train imgsz | deploy input |
| --- | --- | ---: | --- |
| `hswish_640` | Hard-Swish | 640 | 480×640 (baseline) |
| `lrelu_640` | LeakyReLU(0.1015625) | 640 | 480×640 |
| `lrelu_512` | LeakyReLU(0.1015625) | 512 | 384×512 |
| `lrelu_384` | LeakyReLU(0.1015625) | 384 | 288×384 |

* `cpsa_ml.variant_study` holds the training loop; this notebook only sets the
  paths and the matrix.
* Every row starts from the official COCO `yolov8n-pose.pt`: shape-matching
  backbone/neck tensors and the Pose head's box (`cv2`), class (`cv3`) and
  `dfl` branches are copied; only the keypoint branch `cv4` is freshly
  initialised (`<variant>/transfer_report.json` logs what moved).
* All rows share the seed, schedule and augmentations pinned in
  `variant_study.AUGMENTATIONS`.
* Each row trains square at its own `train_imgsz`, then validates and exports
  `deploy.onnx` at its deployment size.
* Outputs go to `/kaggle/working/variant_study/<activation>_<train_imgsz>/`:
  `best.pt`, `last.pt`, `results.csv`, `metrics_fp32.json` (with
  `metrics.half` and `metrics.full`), `transfer_report.json`, `deploy.onnx`.

The INT8/FP32 scoring runs on the host with `tools/vitis/dump_quant_outputs.py`
and `tools/vitis/eval_quant.py`. The accept/reject rule is the proposal in
`ml/README.md` ("Variant study").

In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run.
COMMIT_SHA = "c5f64644f5080f4bfa593159800385826d60233d"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
from pathlib import Path

import torch
import ultralytics

from cpsa_ml.variant_study import DEFAULT_VARIANTS, StudyConfig, train_study

DATA = Path("/kaggle/input/cpsa-halfbust/data.yaml")
FULL_DATA = Path("/kaggle/input/cpsa-fullframe")  # notebook-02 dataset
SCHEMA = "halfbust62"

# Proposed matrix: (activation, train_imgsz, deploy HxW) per row. Replace with
# a subset (for example DEFAULT_VARIANTS[1:]) to run only some rows.
VARIANTS = DEFAULT_VARIANTS

SEED = 0
EPOCHS = 100
BATCH = 32
TRAIN_DEVICE = "0,1"  # 2x T4 DDP
VAL_DEVICE = "0"
WORKERS = 4
PATIENCE = 50
AMP = True

PROJECT = Path("/kaggle/working/runs")
PUBLISH = Path("/kaggle/working/variant_study")

# Official COCO weights, attached to the checkpoints dataset; otherwise
# Ultralytics downloads them (Internet is enabled for this kernel).
COCO_WEIGHTS = Path("/kaggle/input/cpsa-checkpoints/yolov8n-pose.pt")
if not COCO_WEIGHTS.is_file():
    COCO_WEIGHTS = Path("yolov8n-pose.pt")

# Per-row resume, keyed by <activation>_<train_imgsz>:
# {"lrelu_512": "/kaggle/input/cpsa-checkpoints/variant_study/lrelu_512/last.pt"}
RESUME = {}

if not DATA.is_file():
    raise FileNotFoundError(f"attach fabiocabeccia/cpsa-halfbust, missing {DATA}")

config = StudyConfig(
    data=DATA,
    full_data=FULL_DATA,
    coco_weights=COCO_WEIGHTS,
    project=PROJECT,
    publish=PUBLISH,
    schema=SCHEMA,
    variants=VARIANTS,
    resume=RESUME,
    seed=SEED,
    epochs=EPOCHS,
    batch=BATCH,
    device=TRAIN_DEVICE,
    val_device=VAL_DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    amp=AMP,
)
print("ultralytics", ultralytics.__version__, "torch", torch.__version__)
for entry in config.variants:
    print(
        f"{entry.name}: train {entry.train_imgsz}px, "
        f"deploy {entry.deploy_size[0]}x{entry.deploy_size[1]}"
    )

In [ ]:
summaries = train_study(config)
for summary in summaries:
    print(summary["variant"], "->", summary["run_dir"])
    for kind, path in summary["artifacts"].items():
        print(f"  {kind:22s} {path}")

## Next steps

1. Publish `/kaggle/working/variant_study` as a new version of the
   `fabiocabeccia/cpsa-checkpoints` dataset.
2. On the host, quantize and dump the INT8 head tensors for every variant. Read
   `height` and `width` from the variant's `metrics_fp32.json` (`deploy_size`),
   for example for `hswish_640` (480×640):

   ```bash
   VARIANT=hswish_640
   tools/vitis/run_vitis.sh python tools/vitis/dump_quant_outputs.py \
       --weights variant_study/$VARIANT/best.pt \
       --calib-images train=~/data/cpsa-halfbust/images/train \
       --calib-images train_full=~/data/cpsa-fullframe/images/train \
       --test-images half=~/data/cpsa-halfbust/images/val \
       --test-images full=~/data/cpsa-fullframe/images/val \
       --calib-count 64 --test-count 64 --height 480 --width 640 --threads 1 \
       --out build/variant_study/$VARIANT/quant
   ```

3. Score INT8 and FP32 with the same metric code, against the `hswish_640`
   INT8 baseline:

   ```bash
   PYTHONPATH=src python tools/vitis/eval_quant.py \
       --dump build/variant_study/$VARIANT/quant/quant_outputs.npz \
       --weights variant_study/$VARIANT/best.pt \
       --eval-split half=~/data/cpsa-halfbust/images/val:~/data/cpsa-halfbust/labels/val \
       --eval-split full=~/data/cpsa-fullframe/images/val:~/data/cpsa-fullframe/labels/val \
       --baseline build/variant_study/hswish_640/eval/metrics_int8.json \
       --out build/variant_study/$VARIANT/eval
   ```

   `comparison_splits.md` shows the half and full splits side by side; the
   `framing.*` rows say how many people each split really shows as full body.

4. Apply the proposal rule from `ml/README.md`: a faster activation/input
   pairing is accepted only when its INT8 wrist, hand and face PCK@0.2fw stay
   within two points of the INT8 baseline; `comparison.json` reports the
   per-group decision (`pass` / `fail` / `unknown`).